In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load datasets
clinical_df = pd.read_excel('clinical_treatment_data.xlsx')
gene_df = pd.read_excel('gene_mutations.xlsx')

print("Clinical Data Head:")
print(clinical_df.head())
print("\nGene Data Head:")
print(gene_df.head())

Clinical Data Head:
                              bcr                             file_uuid  \
0  Nationwide Children's Hospital  65AB43AF-1C22-441C-9C5C-3AF21A226596   
1  Nationwide Children's Hospital  94B3364B-8977-4F69-BC34-61F669830A53   
2  Nationwide Children's Hospital  37B90D73-F869-4F2F-8A83-5CB58398B8B3   
3  Nationwide Children's Hospital  07AAA163-50FE-4064-ACC2-C127FAB16D2B   
4  Nationwide Children's Hospital  B289F2F4-9F8A-4D36-9884-4CDCD5DCF006   

  batch_number project_code disease_code  day_of_dcc_upload  \
0     379.35.0         TCGA         BRCA                 22   
1     379.35.0         TCGA         BRCA                 22   
2     379.35.0         TCGA         BRCA                 22   
3     379.35.0         TCGA         BRCA                 22   
4     117.79.0         TCGA         BRCA                 22   

   month_of_dcc_upload  year_of_dcc_upload  withdrawn tumor_tissue_site  ...  \
0                   12                2016      False            Breas

In [4]:
print("\n--- Phase 1: Clinical Data Cleaning and Standardization ---")

# Create a copy to avoid modifying the original DataFrame directly
clinical_df_cleaned = clinical_df.copy()

# 1. Convert 'days_to_birth' to 'age_at_diagnosis' (in years)
# 'days_to_birth' is often negative, representing days before a reference date.
# We take the absolute value and divide by 365.25 for years.
clinical_df_cleaned['days_to_birth'] = pd.to_numeric(clinical_df_cleaned['days_to_birth'], errors='coerce')
clinical_df_cleaned['age_at_diagnosis'] = (clinical_df_cleaned['days_to_birth'].abs() / 365.25).round(0)
print("Converted 'days_to_birth' to 'age_at_diagnosis'.")

# 2. Standardize boolean-like columns to 0/1
# Many columns use 'YES'/'NO', 'true'/'false', 'Positive'/'Negative' for boolean flags.
# We map them to 1 (True) and 0 (False) and handle other values as NaN for later imputation.
bool_cols_mapping = {
    'yes': 1, 'no': 0,
    'true': 1, 'false': 0,
    'positive': 1, 'negative': 0,
    'present': 1, 'absent': 0,
    'with tumor': 1, 'tumor free': 0 # For 'person_neoplasm_cancer_status'
}

# List of columns identified as boolean-like from initial data inspection
boolean_like_columns = [
    'withdrawn', 'history_of_neoadjuvant_treatment', 'informed_consent_verified',
    'postoperative_rx_tx', 'radiation_therapy', 'new_tumor_event_after_initial_treatment',
    'lost_follow_up', 'tx_on_clinical_trial', 'therapy_ongoing',
    'radiation_treatment_ongoing', 'distant_metastasis_present_ind2',
    'breast_carcinoma_progesterone_receptor_status', 'breast_carcinoma_estrogen_receptor_status',
    'cytokeratin_immunohistochemistry_staining_method_micrometastasis_indicator',
    'person_neoplasm_cancer_status'
]

for col in boolean_like_columns:
    if col in clinical_df_cleaned.columns:
        clinical_df_cleaned[col] = clinical_df_cleaned[col].astype(str).str.lower().map(bool_cols_mapping).fillna(np.nan)
print("Standardized boolean-like columns to 0/1.")

# 3. Standardize specific categorical entries
# Example: 'menopause_status' has long descriptive strings.
if 'menopause_status' in clinical_df_cleaned.columns:
    clinical_df_cleaned['menopause_status'] = clinical_df_cleaned['menopause_status'].replace({
        'Pre (<6 months since LMP AND no prior bilateral ovariectomy AND not on estrogen replacement)': 'Pre',
        'Post (prior bilateral ovariectomy OR >12 mo since LMP with no prior hysterectomy)': 'Post',
        'Indeterminate (neither Pre or Postmenopausal)': 'Indeterminate (neither Pre or Postmenopausal)' # Keep as is or simplify further if needed
    }).astype(str).str.strip() # Remove leading/trailing whitespace
print("Standardized 'menopause_status' entries.")

# 4. Drop highly sparse or irrelevant columns
# Columns with >70% missing values or administrative IDs that won't be features for RL state.
# This list is determined by inspecting clinical_df.info() and domain knowledge.
cols_to_drop_high_missing_or_irrelevant = [
    'file_uuid', 'batch_number', 'project_code', 'disease_code', 'day_of_dcc_upload',
    'month_of_dcc_upload', 'year_of_dcc_upload', 'other_dx', 'tissue_source_site',
    'patient_id', 'bcr_patient_uuid', 'icd_o_3_site', 'icd_o_3_histology', 'icd_10',
    'year_of_initial_pathologic_diagnosis', 'day_of_form_completion', 'month_of_form_completion',
    'year_of_form_completion', 'bcr_followup_barcode', 'bcr_followup_uuid', 'bcr_drug_barcode',
    'bcr_drug_uuid', 'source_file', 'breast_carcinoma_immunohistochemistry_er_pos_finding_scale',
    'immunohistochemistry_positive_cell_score', 'her2_immunohistochemistry_level_result',
    'her2_erbb_pos_finding_cell_percent_category', 'breast_carcinoma_immunohistochemistry_progesterone_receptor_pos_finding_scale',
    'breast_carcinoma_immunohistochemistry_pos_cell_score', 'histological_type_other',
    'lab_procedure_her2_neu_in_situ_hybrid_outcome_type', 'bcr_radiation_barcode', 'bcr_radiation_uuid',
    'units', 'therapy_type_notes', 'regimen_indication', 'route_of_administration', 'regimen_number',
    'course_number', 'prescribed_dose_units', 'total_dose_units', 'surgical_procedure_purpose_other_text',
    'axillary_lymph_node_stage_other_method_descriptive_text', 'init_pathology_dx_method_other',
    'pos_finding_progesterone_receptor_other_measurement_scale_text',
    'positive_finding_estrogen_receptor_other_measurement_scale_text', 'regimen_indication_notes',
    'followup_case_report_form_submission_reason', 'clinical_trail_drug_classification',
    'metastatic_site_at_diagnosis', 'her2_neu_and_centromere_17_copy_number_analysis_input_total_number_count',
    'her2_erbb_pos_finding_fluorescence_in_situ_hybridization_calculation_method_text',
    'metastatic_site_at_diagnosis_other', 'breast_neoplasm_other_surgical_procedure_descriptive_text',
    'her2_erbb_method_calculation_method_text', 'new_neoplasm_event_type',
    'new_neoplasm_event_occurrence_anatomic_site', 'new_neoplasm_occurrence_anatomic_site_text',
    'days_to_additional_surgery_metastatic_procedure', 'metastatic_breast_carcinoma_estrogen_receptor_status',
    'metastatic_breast_carcinoma_progesterone_receptor_status',
    'metastatic_breast_carcinoma_lab_proc_her2_neu_immunohistochemistry_receptor_status',
    'metastatic_breast_carcinoma_her2_erbb_pos_finding_cell_percent_category',
    'metastatic_breast_carcinoma_erbb2_immunohistochemistry_level_result',
    'new_tumor_event_additional_surgery_procedure', 'days_to_new_tumor_event_additional_surgery_procedure',
    'her2_and_centromere_17_positive_finding_other_measurement_scale_text',
    'metastatic_breast_carcinoma_estrogen_receptor_level_cell_percent_category',
    'er_detection_method_text', 'pgr_detection_method_text',
    'metastatic_breast_carcinoma_immunohistochemistry_er_positive_finding_scale_type',
    'metastatic_breast_carcinoma_immunohistochemistry_progesterone_receptor_positive_finding_scale_type',
    'days_to_last_known_alive', 'metastatic_breast_carcinoma_immunohistochemistry_er_pos_cell_score',
    'metastatic_breast_carcinoma_progesterone_receptor_level_cell_percent_category',
    'metastatic_breast_carcinoma_immunohistochemistry_pr_pos_cell_score',
    'metastatic_breast_carcinoma_her2_erbb_method_calculation_method_text',
    'metastatic_breast_carcinoma_lab_proc_her2_neu_in_situ_hybridization_outcome_type',
    'her2_neu_metastatic_breast_carcinoma_copy_analysis_input_total_number',
    'metastatic_breast_carcinoma_fluorescence_in_situ_hybridization_diagnostic_proc_centromere_17_signal_result_range',
    'her2_neu_and_centromere_17_copy_number_metastatic_breast_carcinoma_analysis_input_total_number_count',
    'metastatic_breast_carcinoma_her2_neu_chromosone_17_signal_ratio_value',
    'metastatic_breast_carcinoma_her2_erbb_pos_finding_fluorescence_in_situ_hybridization_calculation_method_text',
    'metastatic_breast_carcinoma_pos_finding_her2_erbb2_other_measure_scale_text',
    'pos_finding_her2_erbb2_other_measurement_scale_text',
    'pos_finding_metastatic_breast_carcinoma_estrogen_receptor_other_measuremenet_scale_text',
    'metastatic_breast_carcinoma_pos_finding_progesterone_receptor_other_measure_scale_text'
]
clinical_df_cleaned = clinical_df_cleaned.drop(columns=list(set(cols_to_drop_high_missing_or_irrelevant)), errors='ignore')
print(f"Dropped highly sparse/irrelevant columns. New shape: {clinical_df_cleaned.shape}")

# 5. Impute remaining missing values
# Separate numerical and categorical columns for imputation
numerical_cols = clinical_df_cleaned.select_dtypes(include=np.number).columns.tolist()
categorical_cols = clinical_df_cleaned.select_dtypes(include='object').columns.tolist()

# Median imputation for numerical columns
for col in numerical_cols:
    if clinical_df_cleaned[col].isnull().any():
        clinical_df_cleaned[col] = pd.to_numeric(clinical_df_cleaned[col], errors='coerce') # Ensure numeric type
        clinical_df_cleaned[col].fillna(clinical_df_cleaned[col].median(), inplace=True)
print("Imputed numerical columns with median.")

# Mode imputation for categorical columns
for col in categorical_cols:
    if clinical_df_cleaned[col].isnull().any() or clinical_df_cleaned[col].isin(['nan', '', ' ']).any():
        # Replace string 'nan', empty string, or space with actual NaN for proper mode calculation
        clinical_df_cleaned[col] = clinical_df_cleaned[col].astype(str).replace(['nan', '', ' '], np.nan)
        # Fill with mode, if mode is empty (all NaNs), fill with 'Unknown'
        mode_val = clinical_df_cleaned[col].mode()
        if not mode_val.empty:
            clinical_df_cleaned[col].fillna(mode_val, inplace=True)
        else:
            clinical_df_cleaned[col].fillna('Unknown', inplace=True)
print("Imputed categorical columns with mode or 'Unknown'.")

print(f"\nClinical Data after cleaning and initial standardization. Shape: {clinical_df_cleaned.shape}")
print(clinical_df_cleaned.head())
print("\nMissing values after Phase 1:")
print(clinical_df_cleaned.isnull().sum().sum()) # Should be 0


--- Phase 1: Clinical Data Cleaning and Standardization ---
Converted 'days_to_birth' to 'age_at_diagnosis'.
Standardized boolean-like columns to 0/1.
Standardized 'menopause_status' entries.
Dropped highly sparse/irrelevant columns. New shape: (488, 74)
Imputed numerical columns with median.
Imputed categorical columns with mode or 'Unknown'.

Clinical Data after cleaning and initial standardization. Shape: (488, 74)
                              bcr  withdrawn tumor_tissue_site  gender  \
0  Nationwide Children's Hospital          0            Breast  FEMALE   
1  Nationwide Children's Hospital          0            Breast  FEMALE   
2  Nationwide Children's Hospital          0            Breast  FEMALE   
3  Nationwide Children's Hospital          0            Breast  FEMALE   
4  Nationwide Children's Hospital          0            Breast  FEMALE   

  vital_status  days_to_birth  days_to_last_followup  \
0        Alive       -20211.0                 4047.0   
1        Alive      

In [3]:
print("\n--- Phase 2: Feature Engineering from Clinical Data ---")

# 1. Derive Treatment Durations
# Ensure columns are numeric before subtraction
clinical_df_cleaned['days_to_drug_therapy_start'] = pd.to_numeric(clinical_df_cleaned['days_to_drug_therapy_start'], errors='coerce').fillna(0)
clinical_df_cleaned['days_to_drug_therapy_end'] = pd.to_numeric(clinical_df_cleaned['days_to_drug_therapy_end'], errors='coerce').fillna(0)
clinical_df_cleaned['days_to_radiation_therapy_start'] = pd.to_numeric(clinical_df_cleaned['days_to_radiation_therapy_start'], errors='coerce').fillna(0)
clinical_df_cleaned['days_to_radiation_therapy_end'] = pd.to_numeric(clinical_df_cleaned['days_to_radiation_therapy_end'], errors='coerce').fillna(0)

clinical_df_cleaned['treatment_duration_drug'] = (clinical_df_cleaned['days_to_drug_therapy_end'] - clinical_df_cleaned['days_to_drug_therapy_start']).clip(lower=0)
clinical_df_cleaned['treatment_duration_radiation'] = (clinical_df_cleaned['days_to_radiation_therapy_end'] - clinical_df_cleaned['days_to_radiation_therapy_start']).clip(lower=0)
print("Derived 'treatment_duration_drug' and 'treatment_duration_radiation'.")

# 2. Create a unified 'survival_days' outcome
# Prioritize 'days_to_death' if available, otherwise use 'days_to_last_followup'.
clinical_df_cleaned['days_to_death'] = pd.to_numeric(clinical_df_cleaned['days_to_death'], errors='coerce')
clinical_df_cleaned['days_to_last_followup'] = pd.to_numeric(clinical_df_cleaned['days_to_last_followup'], errors='coerce')

clinical_df_cleaned['survival_days'] = clinical_df_cleaned['days_to_death'].fillna(clinical_df_cleaned['days_to_last_followup'])
clinical_df_cleaned['survival_days'].fillna(clinical_df_cleaned['survival_days'].median(), inplace=True) # Impute any remaining NaNs
print("Created 'survival_days' feature.")

# 3. Create Composite Feature: Breast Cancer Subtype
# This is a simplified classification based on ER/PR/HER2 status.
# Assuming ER/PR are 0 (Negative) or 1 (Positive) from Phase 1.
# HER2 status needs careful mapping: 'Positive' (1), 'Negative' (0), 'Equivocal' (0.5 or treat as separate category)
# For simplicity, we'll map Equivocal to 0 (Negative) for subtype classification.
clinical_df_cleaned['lab_proc_her2_neu_immunohistochemistry_receptor_status'] = \
    clinical_df_cleaned['lab_proc_her2_neu_immunohistochemistry_receptor_status'].astype(str).str.lower().map({'positive': 1, 'negative': 0, 'equivocal': 0}).fillna(0)

def get_breast_cancer_subtype(row):
    er = row['breast_carcinoma_estrogen_receptor_status']
    pr = row['breast_carcinoma_progesterone_receptor_status']
    her2 = row['lab_proc_her2_neu_immunohistochemistry_receptor_status']

    
    if er == 0 and pr == 0 and her2 == 0:
        return 'Basal-Like'
    elif er == 0 and pr == 0 and her2 == 1: # HER2-Positive
        return 'HER2-Enriched'
    elif (er == 1 and pr == 1 and her2 == 0):
        return 'Luminal-A'
    elif (er == 1 and her2 == 1):
        return 'Luminal-B'
    else:
        return 'Undefined' 

clinical_df_cleaned['breast_cancer_subtype'] = clinical_df_cleaned.apply(get_breast_cancer_subtype, axis=1)
print("Created 'breast_cancer_subtype' composite feature.")

# Drop the original Basal-Like, HER2-Enriched, Luminal-A, Luminal-B, Undefined columns if they exist
# as we've created our own composite.
subtype_cols_to_drop = ['Basal-Like', 'HER2-Enriched', 'Luminal-A', 'Luminal-B', 'Undefined']
clinical_df_cleaned = clinical_df_cleaned.drop(columns=[col for col in subtype_cols_to_drop if col in clinical_df_cleaned.columns], errors='ignore')

# 4. Group records by patient and treatment start date to define RL time steps
# This is crucial for creating sequential patient trajectories.
# If multiple drugs started on the same day, they form a 'combination' action.
# We need to keep 'bcr_patient_barcode' and 'days_to_drug_therapy_start' for grouping.
# For other features, we take the 'first' value assuming they represent the patient's state at that time point.
# For drug_name and therapy_type, we aggregate unique values to represent combinations.

# Ensure 'bcr_patient_barcode' and 'days_to_drug_therapy_start' are not dropped and are suitable for grouping
if 'bcr_patient_barcode' not in clinical_df_cleaned.columns:
    print("Error: 'bcr_patient_barcode' not found. Cannot group by patient. Please ensure this column exists in your raw data.")
    exit()

# Convert 'days_to_drug_therapy_start' to numeric, fillna with a large number to push missing dates to end if needed, then sort
clinical_df_cleaned['days_to_drug_therapy_start'] = pd.to_numeric(clinical_df_cleaned['days_to_drug_therapy_start'], errors='coerce').fillna(-1) # Use -1 for missing start dates to sort them to the beginning or handle separately

# Sort by patient and then by treatment start date to ensure correct sequence
clinical_df_cleaned.sort_values(by=['bcr_patient_barcode', 'days_to_drug_therapy_start'], inplace=True)

# Define aggregation logic for grouping
aggregation_rules = {
    'drug_name': lambda x: ', '.join(x.dropna().astype(str).unique()) if x.dropna().any() else 'No Drug',
    'therapy_type': lambda x: ', '.join(x.dropna().astype(str).unique()) if x.dropna().any() else 'No Therapy Type',
    'measure_of_response': 'first',
    'vital_status': 'first',
    'survival_days': 'first',
    'new_tumor_event_after_initial_treatment': 'first',
    'radiation_therapy': 'first',
    'radiation_dosage': 'first',
    'numfractions': 'first',
    'treatment_duration_drug': 'first',
    'treatment_duration_radiation': 'first',
    'age_at_diagnosis': 'first',
    'gender': 'first',
    'race': 'first',
    'ethnicity': 'first',
    'menopause_status': 'first',
    'histological_type': 'first',
    'pathologic_stage': 'first',
    'pathologic_T': 'first',
    'pathologic_N': 'first',
    'pathologic_M': 'first',
    'breast_carcinoma_estrogen_receptor_status': 'first',
    'breast_carcinoma_progesterone_receptor_status': 'first',
    'lab_proc_her2_neu_immunohistochemistry_receptor_status': 'first',
    'breast_cancer_subtype': 'first',
    'lymph_node_examined_count': 'first',
    'number_of_lymphnodes_positive_by_he': 'first',
    'number_of_lymphnodes_positive_by_ihc': 'first',
    'person_neoplasm_cancer_status': 'first',
    'primary_lymph_node_presentation_assessment': 'first',
    'anatomic_neoplasm_subdivision': 'first',
    'axillary_lymph_node_stage_method_type': 'first',
    'breast_carcinoma_surgical_procedure_name': 'first',
    'margin_status': 'first',
    'er_level_cell_percentage_category': 'first',
    'progesterone_receptor_level_cell_percent_category': 'first',
    'system_version': 'first',
    'prescribed_dose': 'first',
    'total_dose': 'first',
    'her2_neu_chromosone_17_signal_ratio_value': 'first',
    'fluorescence_in_situ_hybridization_diagnostic_procedure_chromosome_17_signal_result_range': 'first',
    'her2_neu_breast_carcinoma_copy_analysis_input_total_number': 'first',
    'breast_carcinoma_primary_surgical_procedure_name': 'first',
    'breast_cancer_surgery_margin_status': 'first',
    'days_to_new_tumor_event_after_initial_treatment': 'first',
    'additional_surgery_locoregional_procedure': 'first',
    'days_to_additional_surgery_locoregional_procedure': 'first',
    'additional_surgery_metastatic_procedure': 'first',
    'additional_radiation_therapy': 'first',
    'additional_pharmaceutical_therapy': 'first',
}

# Filter aggregation rules to only include columns that exist in the DataFrame
existing_aggregation_rules = {k: v for k, v in aggregation_rules.items() if k in clinical_df_cleaned.columns}

clinical_df_grouped = clinical_df_cleaned.groupby(['bcr_patient_barcode', 'days_to_drug_therapy_start']).agg(
    existing_aggregation_rules
).reset_index()

# Filter out records where 'drug_name' is 'No Drug' after grouping, as these represent no active treatment
clinical_df_grouped = clinical_df_grouped[clinical_df_grouped['drug_name']!= 'No Drug']

print(f"\nClinical Data after feature engineering and grouping. Shape: {clinical_df_grouped.shape}")
print(clinical_df_grouped.head())


--- Phase 2: Feature Engineering from Clinical Data ---
Derived 'treatment_duration_drug' and 'treatment_duration_radiation'.
Created 'survival_days' feature.
Created 'breast_cancer_subtype' composite feature.

Clinical Data after feature engineering and grouping. Shape: (483, 52)
  bcr_patient_barcode  days_to_drug_therapy_start       drug_name  \
0        TCGA-3C-AAAU                      1828.0         Zoladex   
1        TCGA-3C-AALK                       371.0     Trastuzumab   
2        TCGA-4H-AAAK                        27.0  5-Fluorouracil   
3        TCGA-5T-A9QA                       115.0         Cytoxan   
4        TCGA-A1-A0SF                        65.0              tc   

   therapy_type measure_of_response vital_status  survival_days  \
0  Chemotherapy   Complete Response        Alive          983.5   
1  Chemotherapy                None        Alive          983.5   
2  Chemotherapy   Complete Response        Alive          983.5   
3  Chemotherapy   Complete Respons

In [5]:
print("\n--- Phase 3: Genomic Data Preprocessing and Integration ---")

# 1. Transpose Gene Data
# Set 'attrib_name' as index, then transpose to have patient IDs as rows and gene mutations as columns.
gene_df_transposed = gene_df.set_index('attrib_name').T
gene_df_transposed.index.name = 'patient_barcode_full' # Rename index for clarity
gene_df_transposed = gene_df_transposed.reset_index()
print("Transposed gene data.")

# 2. Standardize Patient IDs for Merging
# The clinical data uses 'TCGA-XX-XXXX' format for 'bcr_patient_barcode'.
# The gene data uses 'TCGA-XX-XXXX-YY' format for column names (now rows after transpose).
# We need to extract the common 'TCGA-XX-XXXX' part for accurate merging.
gene_df_transposed['bcr_patient_barcode_short'] = gene_df_transposed['patient_barcode_full'].apply(lambda x: '-'.join(x.split('-')[:3]))
clinical_df_grouped['bcr_patient_barcode_short'] = clinical_df_grouped['bcr_patient_barcode'].apply(lambda x: '-'.join(x.split('-')[:3]))
print("Standardized patient barcodes for merging.")

# 3. Merge Clinical and Gene Data
# Perform an inner merge to keep only patients present in both datasets.
merged_df = pd.merge(clinical_df_grouped, gene_df_transposed,
                     on='bcr_patient_barcode_short', how='inner', suffixes=('_clinical', '_gene'))

# Drop the temporary short barcode column and the full gene barcode column as they are no longer needed
merged_df = merged_df.drop(columns=['bcr_patient_barcode_short', 'patient_barcode_full'], errors='ignore')
print(f"Merged clinical and gene data. New shape: {merged_df.shape}")
print(merged_df.head())


--- Phase 3: Genomic Data Preprocessing and Integration ---
Transposed gene data.


NameError: name 'clinical_df_grouped' is not defined

In [5]:
print("\n--- Phase 4: Feature Selection and Final Data Preparation for RL ---")

# 1. Feature Selection for Gene Mutations
# Identify gene mutation columns (these are all columns that were originally in gene_df_transposed, excluding the barcode)
# We need to be careful here to only select the actual gene columns from the merged_df.
# A simple way is to get all columns from gene_df_transposed (excluding its barcode)
# and check if they exist in merged_df.
gene_mutation_cols = [col for col in gene_df_transposed.columns if col not in ['patient_barcode_full', 'bcr_patient_barcode_short']]
gene_mutation_cols_in_merged = [col for col in gene_mutation_cols if col in merged_df.columns]

# Apply Variance Thresholding to gene mutation features
# This removes features (gene mutations) that have very low variance, meaning they are present in almost all
# or almost no patients. Such features provide little discriminatory power.
# A threshold of 0.05 means a feature must have a variance greater than 0.05 to be kept.
# For binary data (0/1), variance = p * (1-p). If p=0.05 or p=0.95, variance is 0.05 * 0.95 = 0.0475.
# So, a threshold of 0.05 will remove features present in <5% or >95% of samples.
selector = VarianceThreshold(threshold=0.05)
# Ensure gene columns are numeric (0 or 1)
merged_df[gene_mutation_cols_in_merged] = merged_df[gene_mutation_cols_in_merged].apply(pd.to_numeric, errors='coerce').fillna(0)

# Fit and transform only the gene mutation columns
gene_features_selected_array = selector.fit_transform(merged_df[gene_mutation_cols_in_merged])

# Get names of selected gene features
selected_gene_cols = merged_df[gene_mutation_cols_in_merged].columns[selector.get_support()]
print(f"Reduced gene features from {len(gene_mutation_cols_in_merged)} to {len(selected_gene_cols)} using Variance Thresholding.")

# Create a DataFrame for selected gene features
df_selected_genes = pd.DataFrame(gene_features_selected_array, columns=selected_gene_cols, index=merged_df.index)

# Drop original gene columns and concatenate with selected ones
final_df_for_rl = merged_df.drop(columns=gene_mutation_cols_in_merged, errors='ignore')
final_df_for_rl = pd.concat([final_df_for_rl, df_selected_genes], axis=1)

# 2. Define Features for RL State, Action, and Reward
# Identify columns that will form the RL state, action, and reward.
# Exclude identifiers and columns that are direct outcomes or will be transformed into action/reward.

# Columns to be used as features for the RL state
state_features_candidate = + list(selected_gene_cols) # Add selected gene features

# Filter to only include columns that actually exist in the DataFrame
state_features = [col for col in state_features_candidate if col in final_df_for_rl.columns]

# Columns for RL Action and Reward
action_col = 'drug_name' # This will be the action
reward_cols = ['measure_of_response', 'vital_status', 'survival_days', 'new_tumor_event_after_initial_treatment'] # For reward calculation
time_col = 'days_to_drug_therapy_start' # For ordering trajectories
patient_id_col = 'bcr_patient_barcode' # Patient identifier

# Ensure essential columns are present
for col in [patient_id_col, time_col, action_col] + reward_cols:
    if col not in final_df_for_rl.columns:
        print(f"Error: Essential column '{col}' missing from final DataFrame. Cannot proceed.")
        exit()

# 3. Encode Categorical Features and Scale Numerical Features for the State
# Identify numerical and categorical features from the final state_features list
numerical_features_for_scaling = final_df_for_rl[state_features].select_dtypes(include=np.number).columns.tolist()
categorical_features_for_encoding = final_df_for_rl[state_features].select_dtypes(include='object').columns.tolist()

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=)

categorical_transformer = Pipeline(steps=)

# Create a ColumnTransformer to apply different transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features_for_scaling),
        ('cat', categorical_transformer, categorical_features_for_encoding)
    ],
    remainder='passthrough' # Keep other columns (like gene features which are already 0/1)
)

# Apply transformations to the selected state features
# Temporarily combine state features with patient_id and time_col for processing
df_to_transform = final_df_for_rl[[patient_id_col, time_col] + state_features].copy()
df_to_transform_indexed = df_to_transform.set_index([patient_id_col, time_col])

# Fit and transform the data
processed_state_features_array = preprocessor.fit_transform(df_to_transform_indexed)

# Get feature names for the transformed data
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features_for_encoding)
all_state_feature_names = numerical_features_for_scaling + list(ohe_feature_names)

# Add back any 'passthrough' columns (which would be the selected gene features)
passthrough_cols = [col for col in state_features if col not in numerical_features_for_scaling and col not in categorical_features_for_encoding]
all_state_feature_names.extend(passthrough_cols)

# Create a DataFrame with transformed state features
processed_state_df = pd.DataFrame(processed_state_features_array, columns=all_state_feature_names, index=df_to_transform_indexed.index)
processed_state_df = processed_state_df.reset_index() # Bring back patient_id and time_col

print(f"\nProcessed State Features DataFrame. Shape: {processed_state_df.shape}")
print(processed_state_df.head())

# 4. Define Action Space and Map Actions to Integers
all_drug_combinations = final_df_for_rl[action_col].unique()
action_to_int = {action: i for i, action in enumerate(all_drug_combinations)}
int_to_action = {i: action for action, i in action_to_int.items()} # For reverse mapping

final_df_for_rl['action_id'] = final_df_for_rl[action_col].map(action_to_int)
print(f"\nDefined {len(all_drug_combinations)} unique actions (drug combinations).")

# 5. Define Reward Function
def calculate_reward(row):
    reward = 0

    # Tumor Response (from 'measure_of_response')
    # Higher positive reward for better response
    if row['measure_of_response'] == 'Complete Response':
        reward += 5
    elif row['measure_of_response'] == 'Partial Response':
        reward += 3
    elif row['measure_of_response'] == 'Stable Disease':
        reward += 1
    elif row['measure_of_response'] == 'Progressive Disease':
        reward -= 5 # Significant penalty for progression
    else: # Handle 'Unknown' or other unmapped responses
        reward -= 1 # Small penalty for unclear response

    # Survival (from 'vital_status' and 'survival_days')
    # Long-term positive reward for being alive, penalty for death.
    # Normalize survival days to contribute proportionally to reward.
    max_survival = final_df_for_rl['survival_days'].max()
    min_survival = final_df_for_rl['survival_days'].min()
    if max_survival > min_survival:
        normalized_survival = (row['survival_days'] - min_survival) / (max_survival - min_survival)
        reward += normalized_survival * 2 # Smaller weight than immediate response
    else:
        reward += 1 # Neutral if no variation

    if row['vital_status'] == 'Dead':
        reward -= 10 # Large penalty for death

    # Disease Progression/Recurrence (from 'new_tumor_event_after_initial_treatment')
    # Penalty for new tumor events
    if row['new_tumor_event_after_initial_treatment'] == 1: # Assuming 1 for YES
        reward -= 7 # Penalty for recurrence

    # Toxicity (Implicit, as direct toxicity data is limited)
    # This is a placeholder. In a real scenario, explicit toxicity grades would be used.
    # For now, we can infer some toxicity from very short treatment durations or early death/progression
    # if not already captured by other penalties.
    # This part is more conceptual for this dataset due to lack of explicit toxicity data.
    # If treatment duration is very short and outcome is poor, it might imply toxicity/intolerance.
    if row['treatment_duration_drug'] < 30 and (row['measure_of_response'] == 'Progressive Disease' or row['vital_status'] == 'Dead'):
        reward -= 2 # Small additional penalty for very short, ineffective/harmful treatment

    return reward

final_df_for_rl['reward'] = final_df_for_rl.apply(calculate_reward, axis=1)
print("Calculated 'reward' for each treatment step.")

# 6. Construct Patient Trajectories for RL Training
# Merge processed state features back with action_id, reward, and original patient_id/time_col
# This ensures all relevant data for a trajectory step is together.
rl_data_full = pd.merge(
    processed_state_df,
    final_df_for_rl[[patient_id_col, time_col, 'action_id', 'reward']],
    on=[patient_id_col, time_col],
    how='inner'
)

# Sort again to ensure correct temporal order for trajectories
rl_data_full.sort_values(by=[patient_id_col, time_col], inplace=True)

patient_trajectories = defaultdict(list)
# Iterate through each patient to build their sequence of (state, action, reward, next_state, done)
for patient_id in rl_data_full[patient_id_col].unique():
    patient_records = rl_data_full[rl_data_full[patient_id_col] == patient_id].copy()
    
    # Extract state features (all columns except patient_id, time_col, action_id, reward)
    current_state_feature_cols = [col for col in processed_state_df.columns if col not in [patient_id_col, time_col]]

    for i in range(len(patient_records)):
        current_step = patient_records.iloc[i]
        current_state = current_step[current_state_feature_cols].values
        action = current_step['action_id']
        reward = current_step['reward']
        
        next_state = None
        done = False
        if i + 1 < len(patient_records):
            next_step = patient_records.iloc[i+1]
            next_state = next_step[current_state_feature_cols].values
        else:
            done = True # This is the last observed step for this patient

        patient_trajectories[patient_id].append({
            'state': current_state,
            'action': action,
            'reward': reward,
            'next_state': next_state,
            'done': done
        })

print(f"\nConstructed {len(patient_trajectories)} patient trajectories.")
print("\nExample of the first step for the first patient's trajectory:")
if patient_trajectories:
    first_patient_id = list(patient_trajectories.keys())
    first_step = patient_trajectories[first_patient_id]
    print(f"  Patient ID: {first_patient_id}")
    print(f"  State (first 5 values): {first_step['state'][:5]}")
    print(f"  Action ID: {first_step['action']} (Drug: {int_to_action.get(first_step['action'], 'Unknown')})")
    print(f"  Reward: {first_step['reward']:.2f}")
    print(f"  Is Done: {first_step['done']}")
    if first_step['next_state'] is not None:
        print(f"  Next State (first 5 values): {first_step['next_state'][:5]}")
else:
    print("No trajectories could be constructed. Check data integrity and merging steps.")

print("\n--- Preprocessing Complete ---")

SyntaxError: invalid syntax (<ipython-input-5-eeb503ded953>, line 63)

In [6]:
print("\n--- Phase 4: Feature Selection and Final Data Preparation for RL ---")
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Feature Selection for Gene Mutations
# Identify gene mutation columns (these are all columns that were originally in gene_df_transposed, excluding the barcode)
# We need to be careful here to only select the actual gene columns from the merged_df.
# A simple way is to get all columns from gene_df_transposed (excluding its barcode)
# and check if they exist in merged_df.
gene_mutation_cols = [col for col in gene_df_transposed.columns if col not in ['patient_barcode_full', 'bcr_patient_barcode_short']]
gene_mutation_cols_in_merged = [col for col in gene_mutation_cols if col in merged_df.columns]

# Apply Variance Thresholding to gene mutation features
# This removes features (gene mutations) that have very low variance, meaning they are present in almost all
# or almost no patients. Such features provide little discriminatory power.
# A threshold of 0.05 means a feature must have a variance greater than 0.05 to be kept.
# For binary data (0/1), variance = p * (1-p). If p=0.05 or p=0.95, variance is 0.05 * 0.95 = 0.0475.
# So, a threshold of 0.05 will remove features present in <5% or >95% of samples.
selector = VarianceThreshold(threshold=0.05)
# Ensure gene columns are numeric (0 or 1)
merged_df[gene_mutation_cols_in_merged] = merged_df[gene_mutation_cols_in_merged].apply(pd.to_numeric, errors='coerce').fillna(0)

# Fit and transform only the gene mutation columns
gene_features_selected_array = selector.fit_transform(merged_df[gene_mutation_cols_in_merged])

# Get names of selected gene features
selected_gene_cols = merged_df[gene_mutation_cols_in_merged].columns[selector.get_support()]
print(f"Reduced gene features from {len(gene_mutation_cols_in_merged)} to {len(selected_gene_cols)} using Variance Thresholding.")

# Create a DataFrame for selected gene features
df_selected_genes = pd.DataFrame(gene_features_selected_array, columns=selected_gene_cols, index=merged_df.index)

# Drop original gene columns and concatenate with selected ones
final_df_for_rl = merged_df.drop(columns=gene_mutation_cols_in_merged, errors='ignore')
final_df_for_rl = pd.concat([final_df_for_rl, df_selected_genes], axis=1)

# 2. Define Features for RL State, Action, and Reward
# Identify columns that will form the RL state, action, and reward.
# Exclude identifiers and columns that are direct outcomes or will be transformed into action/reward.

# Columns to be used as features for the RL state
state_features_candidate = + list(selected_gene_cols) # Add selected gene features

# Filter to only include columns that actually exist in the DataFrame
state_features = [col for col in state_features_candidate if col in final_df_for_rl.columns]

# Columns for RL Action and Reward
action_col = 'drug_name' # This will be the action
reward_cols = ['measure_of_response', 'vital_status', 'survival_days', 'new_tumor_event_after_initial_treatment'] # For reward calculation
time_col = 'days_to_drug_therapy_start' # For ordering trajectories
patient_id_col = 'bcr_patient_barcode' # Patient identifier

# Ensure essential columns are present
for col in [patient_id_col, time_col, action_col] + reward_cols:
    if col not in final_df_for_rl.columns:
        print(f"Error: Essential column '{col}' missing from final DataFrame. Cannot proceed.")
        exit()

# 3. Encode Categorical Features and Scale Numerical Features for the State
# Identify numerical and categorical features from the final state_features list
numerical_features_for_scaling = final_df_for_rl[state_features].select_dtypes(include=np.number).columns.tolist()
categorical_features_for_encoding = final_df_for_rl[state_features].select_dtypes(include='object').columns.tolist()

# Create preprocessing pipelines for numerical and categorical features
# Define actual transformers
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

# Create a ColumnTransformer to apply different transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features_for_scaling),
        ('cat', categorical_transformer, categorical_features_for_encoding)
    ],
    remainder='passthrough' # Keep other columns (like gene features which are already 0/1)
)

# Apply transformations to the selected state features
# Temporarily combine state features with patient_id and time_col for processing
df_to_transform = final_df_for_rl[[patient_id_col, time_col] + state_features].copy()
df_to_transform_indexed = df_to_transform.set_index([patient_id_col, time_col])

# Fit and transform the data
processed_state_features_array = preprocessor.fit_transform(df_to_transform_indexed)

# Get feature names for the transformed data
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features_for_encoding)
all_state_feature_names = numerical_features_for_scaling + list(ohe_feature_names)

# Add back any 'passthrough' columns (which would be the selected gene features)
passthrough_cols = [col for col in state_features if col not in numerical_features_for_scaling and col not in categorical_features_for_encoding]
all_state_feature_names.extend(passthrough_cols)

# Create a DataFrame with transformed state features
processed_state_df = pd.DataFrame(processed_state_features_array, columns=all_state_feature_names, index=df_to_transform_indexed.index)
processed_state_df = processed_state_df.reset_index() # Bring back patient_id and time_col

print(f"\nProcessed State Features DataFrame. Shape: {processed_state_df.shape}")
print(processed_state_df.head())

# 4. Define Action Space and Map Actions to Integers
all_drug_combinations = final_df_for_rl[action_col].unique()
action_to_int = {action: i for i, action in enumerate(all_drug_combinations)}
int_to_action = {i: action for action, i in action_to_int.items()} # For reverse mapping

final_df_for_rl['action_id'] = final_df_for_rl[action_col].map(action_to_int)
print(f"\nDefined {len(all_drug_combinations)} unique actions (drug combinations).")

# 5. Define Reward Function
def calculate_reward(row):
    reward = 0

    # Tumor Response (from 'measure_of_response')
    # Higher positive reward for better response
    if row['measure_of_response'] == 'Complete Response':
        reward += 5
    elif row['measure_of_response'] == 'Partial Response':
        reward += 3
    elif row['measure_of_response'] == 'Stable Disease':
        reward += 1
    elif row['measure_of_response'] == 'Progressive Disease':
        reward -= 5 # Significant penalty for progression
    else: # Handle 'Unknown' or other unmapped responses
        reward -= 1 # Small penalty for unclear response

    # Survival (from 'vital_status' and 'survival_days')
    # Long-term positive reward for being alive, penalty for death.
    # Normalize survival days to contribute proportionally to reward.
    max_survival = final_df_for_rl['survival_days'].max()
    min_survival = final_df_for_rl['survival_days'].min()
    if max_survival > min_survival:
        normalized_survival = (row['survival_days'] - min_survival) / (max_survival - min_survival)
        reward += normalized_survival * 2 # Smaller weight than immediate response
    else:
        reward += 1 # Neutral if no variation

    if row['vital_status'] == 'Dead':
        reward -= 10 # Large penalty for death

    # Disease Progression/Recurrence (from 'new_tumor_event_after_initial_treatment')
    # Penalty for new tumor events
    if row['new_tumor_event_after_initial_treatment'] == 1: # Assuming 1 for YES
        reward -= 7 # Penalty for recurrence

    # Toxicity (Implicit, as direct toxicity data is limited)
    # This is a placeholder. In a real scenario, explicit toxicity grades would be used.
    # For now, we can infer some toxicity from very short treatment durations or early death/progression
    # if not already captured by other penalties.
    # This part is more conceptual for this dataset due to lack of explicit toxicity data.
    # If treatment duration is very short and outcome is poor, it might imply toxicity/intolerance.
    if row['treatment_duration_drug'] < 30 and (row['measure_of_response'] == 'Progressive Disease' or row['vital_status'] == 'Dead'):
        reward -= 2 # Small additional penalty for very short, ineffective/harmful treatment

    return reward

final_df_for_rl['reward'] = final_df_for_rl.apply(calculate_reward, axis=1)
print("Calculated 'reward' for each treatment step.")

# 6. Construct Patient Trajectories for RL Training
# Merge processed state features back with action_id, reward, and original patient_id/time_col
# This ensures all relevant data for a trajectory step is together.
rl_data_full = pd.merge(
    processed_state_df,
    final_df_for_rl[[patient_id_col, time_col, 'action_id', 'reward']],
    on=[patient_id_col, time_col],
    how='inner'
)

# Sort again to ensure correct temporal order for trajectories
rl_data_full.sort_values(by=[patient_id_col, time_col], inplace=True)

patient_trajectories = defaultdict(list)
# Iterate through each patient to build their sequence of (state, action, reward, next_state, done)
for patient_id in rl_data_full[patient_id_col].unique():
    patient_records = rl_data_full[rl_data_full[patient_id_col] == patient_id].copy()
    
    # Extract state features (all columns except patient_id, time_col, action_id, reward)
    current_state_feature_cols = [col for col in processed_state_df.columns if col not in [patient_id_col, time_col]]

    for i in range(len(patient_records)):
        current_step = patient_records.iloc[i]
        current_state = current_step[current_state_feature_cols].values
        action = current_step['action_id']
        reward = current_step['reward']
        
        next_state = None
        done = False
        if i + 1 < len(patient_records):
            next_step = patient_records.iloc[i+1]
            next_state = next_step[current_state_feature_cols].values
        else:
            done = True # This is the last observed step for this patient

        patient_trajectories[patient_id].append({
            'state': current_state,
            'action': action,
            'reward': reward,
            'next_state': next_state,
            'done': done
        })

print(f"\nConstructed {len(patient_trajectories)} patient trajectories.")
print("\nExample of the first step for the first patient's trajectory:")
if patient_trajectories:
    first_patient_id = list(patient_trajectories.keys())
    first_step = patient_trajectories[first_patient_id]
    print(f"  Patient ID: {first_patient_id}")
    print(f"  State (first 5 values): {first_step['state'][:5]}")
    print(f"  Action ID: {first_step['action']} (Drug: {int_to_action.get(first_step['action'], 'Unknown')})")
    print(f"  Reward: {first_step['reward']:.2f}")
    print(f"  Is Done: {first_step['done']}")
    if first_step['next_state'] is not None:
        print(f"  Next State (first 5 values): {first_step['next_state'][:5]}")
else:
    print("No trajectories could be constructed. Check data integrity and merging steps.")

print("\n--- Preprocessing Complete ---")


--- Phase 4: Feature Selection and Final Data Preparation for RL ---
Reduced gene features from 101 to 2 using Variance Thresholding.


TypeError: bad operand type for unary +: 'list'

In [7]:
print("\n--- Phase 4: Feature Selection and Final Data Preparation for RL ---")
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Feature Selection for Gene Mutations
# Identify gene mutation columns (these are all columns that were originally in gene_df_transposed, excluding the barcode)
# We need to be careful here to only select the actual gene columns from the merged_df.
# A simple way is to get all columns from gene_df_transposed (excluding its barcode)
# and check if they exist in merged_df.
gene_mutation_cols = [col for col in gene_df_transposed.columns if col not in ['patient_barcode_full', 'bcr_patient_barcode_short']]
gene_mutation_cols_in_merged = [col for col in gene_mutation_cols if col in merged_df.columns]

# Apply Variance Thresholding to gene mutation features
# This removes features (gene mutations) that have very low variance, meaning they are present in almost all
# or almost no patients. Such features provide little discriminatory power.
# A threshold of 0.05 means a feature must have a variance greater than 0.05 to be kept.
# For binary data (0/1), variance = p * (1-p). If p=0.05 or p=0.95, variance is 0.05 * 0.95 = 0.0475.
# So, a threshold of 0.05 will remove features present in <5% or >95% of samples.
selector = VarianceThreshold(threshold=0.05)
# Ensure gene columns are numeric (0 or 1)
merged_df[gene_mutation_cols_in_merged] = merged_df[gene_mutation_cols_in_merged].apply(pd.to_numeric, errors='coerce').fillna(0)

# Fit and transform only the gene mutation columns
gene_features_selected_array = selector.fit_transform(merged_df[gene_mutation_cols_in_merged])

# Get names of selected gene features
selected_gene_cols = merged_df[gene_mutation_cols_in_merged].columns[selector.get_support()]
print(f"Reduced gene features from {len(gene_mutation_cols_in_merged)} to {len(selected_gene_cols)} using Variance Thresholding.")

# Create a DataFrame for selected gene features
df_selected_genes = pd.DataFrame(gene_features_selected_array, columns=selected_gene_cols, index=merged_df.index)

# Drop original gene columns and concatenate with selected ones
final_df_for_rl = merged_df.drop(columns=gene_mutation_cols_in_merged, errors='ignore')
final_df_for_rl = pd.concat([final_df_for_rl, df_selected_genes], axis=1)

# 2. Define Features for RL State, Action, and Reward
# Identify columns that will form the RL state, action, and reward.
# Exclude identifiers and columns that are direct outcomes or will be transformed into action/reward.

# Columns to be used as features for the RL state
state_features_candidate = list(selected_gene_cols) # Add selected gene features

# Filter to only include columns that actually exist in the DataFrame
state_features = [col for col in state_features_candidate if col in final_df_for_rl.columns]

# Columns for RL Action and Reward
action_col = 'drug_name' # This will be the action
reward_cols = ['measure_of_response', 'vital_status', 'survival_days', 'new_tumor_event_after_initial_treatment'] # For reward calculation
time_col = 'days_to_drug_therapy_start' # For ordering trajectories
patient_id_col = 'bcr_patient_barcode' # Patient identifier

# Ensure essential columns are present
for col in [patient_id_col, time_col, action_col] + reward_cols:
    if col not in final_df_for_rl.columns:
        print(f"Error: Essential column '{col}' missing from final DataFrame. Cannot proceed.")
        exit()

# 3. Encode Categorical Features and Scale Numerical Features for the State
# Identify numerical and categorical features from the final state_features list
numerical_features_for_scaling = final_df_for_rl[state_features].select_dtypes(include=np.number).columns.tolist()
categorical_features_for_encoding = final_df_for_rl[state_features].select_dtypes(include='object').columns.tolist()

# Create preprocessing pipelines for numerical and categorical features
# Define actual transformers
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

# Create a ColumnTransformer to apply different transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features_for_scaling),
        ('cat', categorical_transformer, categorical_features_for_encoding)
    ],
    remainder='passthrough' # Keep other columns (like gene features which are already 0/1)
)

# Apply transformations to the selected state features
# Temporarily combine state features with patient_id and time_col for processing
df_to_transform = final_df_for_rl[[patient_id_col, time_col] + state_features].copy()
df_to_transform_indexed = df_to_transform.set_index([patient_id_col, time_col])

# Fit and transform the data
processed_state_features_array = preprocessor.fit_transform(df_to_transform_indexed)

# Get feature names for the transformed data
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features_for_encoding)
all_state_feature_names = numerical_features_for_scaling + list(ohe_feature_names)

# Add back any 'passthrough' columns (which would be the selected gene features)
passthrough_cols = [col for col in state_features if col not in numerical_features_for_scaling and col not in categorical_features_for_encoding]
all_state_feature_names.extend(passthrough_cols)

# Create a DataFrame with transformed state features
processed_state_df = pd.DataFrame(processed_state_features_array, columns=all_state_feature_names, index=df_to_transform_indexed.index)
processed_state_df = processed_state_df.reset_index() # Bring back patient_id and time_col

print(f"\nProcessed State Features DataFrame. Shape: {processed_state_df.shape}")
print(processed_state_df.head())

# 4. Define Action Space and Map Actions to Integers
all_drug_combinations = final_df_for_rl[action_col].unique()
action_to_int = {action: i for i, action in enumerate(all_drug_combinations)}
int_to_action = {i: action for action, i in action_to_int.items()} # For reverse mapping

final_df_for_rl['action_id'] = final_df_for_rl[action_col].map(action_to_int)
print(f"\nDefined {len(all_drug_combinations)} unique actions (drug combinations).")

# 5. Define Reward Function
def calculate_reward(row):
    reward = 0

    # Tumor Response (from 'measure_of_response')
    # Higher positive reward for better response
    if row['measure_of_response'] == 'Complete Response':
        reward += 5
    elif row['measure_of_response'] == 'Partial Response':
        reward += 3
    elif row['measure_of_response'] == 'Stable Disease':
        reward += 1
    elif row['measure_of_response'] == 'Progressive Disease':
        reward -= 5 # Significant penalty for progression
    else: # Handle 'Unknown' or other unmapped responses
        reward -= 1 # Small penalty for unclear response

    # Survival (from 'vital_status' and 'survival_days')
    # Long-term positive reward for being alive, penalty for death.
    # Normalize survival days to contribute proportionally to reward.
    max_survival = final_df_for_rl['survival_days'].max()
    min_survival = final_df_for_rl['survival_days'].min()
    if max_survival > min_survival:
        normalized_survival = (row['survival_days'] - min_survival) / (max_survival - min_survival)
        reward += normalized_survival * 2 # Smaller weight than immediate response
    else:
        reward += 1 # Neutral if no variation

    if row['vital_status'] == 'Dead':
        reward -= 10 # Large penalty for death

    # Disease Progression/Recurrence (from 'new_tumor_event_after_initial_treatment')
    # Penalty for new tumor events
    if row['new_tumor_event_after_initial_treatment'] == 1: # Assuming 1 for YES
        reward -= 7 # Penalty for recurrence

    # Toxicity (Implicit, as direct toxicity data is limited)
    # This is a placeholder. In a real scenario, explicit toxicity grades would be used.
    # For now, we can infer some toxicity from very short treatment durations or early death/progression
    # if not already captured by other penalties.
    # This part is more conceptual for this dataset due to lack of explicit toxicity data.
    # If treatment duration is very short and outcome is poor, it might imply toxicity/intolerance.
    if row['treatment_duration_drug'] < 30 and (row['measure_of_response'] == 'Progressive Disease' or row['vital_status'] == 'Dead'):
        reward -= 2 # Small additional penalty for very short, ineffective/harmful treatment

    return reward

final_df_for_rl['reward'] = final_df_for_rl.apply(calculate_reward, axis=1)
print("Calculated 'reward' for each treatment step.")

# 6. Construct Patient Trajectories for RL Training
# Merge processed state features back with action_id, reward, and original patient_id/time_col
# This ensures all relevant data for a trajectory step is together.
rl_data_full = pd.merge(
    processed_state_df,
    final_df_for_rl[[patient_id_col, time_col, 'action_id', 'reward']],
    on=[patient_id_col, time_col],
    how='inner'
)

# Sort again to ensure correct temporal order for trajectories
rl_data_full.sort_values(by=[patient_id_col, time_col], inplace=True)

patient_trajectories = defaultdict(list)
# Iterate through each patient to build their sequence of (state, action, reward, next_state, done)
for patient_id in rl_data_full[patient_id_col].unique():
    patient_records = rl_data_full[rl_data_full[patient_id_col] == patient_id].copy()
    
    # Extract state features (all columns except patient_id, time_col, action_id, reward)
    current_state_feature_cols = [col for col in processed_state_df.columns if col not in [patient_id_col, time_col]]

    for i in range(len(patient_records)):
        current_step = patient_records.iloc[i]
        current_state = current_step[current_state_feature_cols].values
        action = current_step['action_id']
        reward = current_step['reward']
        
        next_state = None
        done = False
        if i + 1 < len(patient_records):
            next_step = patient_records.iloc[i+1]
            next_state = next_step[current_state_feature_cols].values
        else:
            done = True # This is the last observed step for this patient

        patient_trajectories[patient_id].append({
            'state': current_state,
            'action': action,
            'reward': reward,
            'next_state': next_state,
            'done': done
        })

print(f"\nConstructed {len(patient_trajectories)} patient trajectories.")
print("\nExample of the first step for the first patient's trajectory:")
if patient_trajectories:
    first_patient_id = list(patient_trajectories.keys())
    first_step = patient_trajectories[first_patient_id]
    print(f"  Patient ID: {first_patient_id}")
    print(f"  State (first 5 values): {first_step['state'][:5]}")
    print(f"  Action ID: {first_step['action']} (Drug: {int_to_action.get(first_step['action'], 'Unknown')})")
    print(f"  Reward: {first_step['reward']:.2f}")
    print(f"  Is Done: {first_step['done']}")
    if first_step['next_state'] is not None:
        print(f"  Next State (first 5 values): {first_step['next_state'][:5]}")
else:
    print("No trajectories could be constructed. Check data integrity and merging steps.")

print("\n--- Preprocessing Complete ---")


--- Phase 4: Feature Selection and Final Data Preparation for RL ---
Reduced gene features from 101 to 2 using Variance Thresholding.


NotFittedError: This OneHotEncoder instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [8]:
print("\n--- Phase 4: Feature Selection and Final Data Preparation for RL ---")
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Feature Selection for Gene Mutations
# Identify gene mutation columns (these are all columns that were originally in gene_df_transposed, excluding the barcode)
# We need to be careful here to only select the actual gene columns from the merged_df.
# A simple way is to get all columns from gene_df_transposed (excluding its barcode)
# and check if they exist in merged_df.
gene_mutation_cols = [col for col in gene_df_transposed.columns if col not in ['patient_barcode_full', 'bcr_patient_barcode_short']]
gene_mutation_cols_in_merged = [col for col in gene_mutation_cols if col in merged_df.columns]

# Apply Variance Thresholding to gene mutation features
# This removes features (gene mutations) that have very low variance, meaning they are present in almost all
# or almost no patients. Such features provide little discriminatory power.
# A threshold of 0.05 means a feature must have a variance greater than 0.05 to be kept.
# For binary data (0/1), variance = p * (1-p). If p=0.05 or p=0.95, variance is 0.05 * 0.95 = 0.0475.
# So, a threshold of 0.05 will remove features present in <5% or >95% of samples.
selector = VarianceThreshold(threshold=0.05)
# Ensure gene columns are numeric (0 or 1)
merged_df[gene_mutation_cols_in_merged] = merged_df[gene_mutation_cols_in_merged].apply(pd.to_numeric, errors='coerce').fillna(0)

# Fit and transform only the gene mutation columns
gene_features_selected_array = selector.fit_transform(merged_df[gene_mutation_cols_in_merged])

# Get names of selected gene features
selected_gene_cols = merged_df[gene_mutation_cols_in_merged].columns[selector.get_support()]
print(f"Reduced gene features from {len(gene_mutation_cols_in_merged)} to {len(selected_gene_cols)} using Variance Thresholding.")

# Create a DataFrame for selected gene features
df_selected_genes = pd.DataFrame(gene_features_selected_array, columns=selected_gene_cols, index=merged_df.index)

# Drop original gene columns and concatenate with selected ones
final_df_for_rl = merged_df.drop(columns=gene_mutation_cols_in_merged, errors='ignore')
final_df_for_rl = pd.concat([final_df_for_rl, df_selected_genes], axis=1)

# 2. Define Features for RL State, Action, and Reward
# Identify columns that will form the RL state, action, and reward.
# Exclude identifiers and columns that are direct outcomes or will be transformed into action/reward.

# Columns to be used as features for the RL state
state_features_candidate = list(selected_gene_cols) # Add selected gene features

# Filter to only include columns that actually exist in the DataFrame
state_features = [col for col in state_features_candidate if col in final_df_for_rl.columns]

# Columns for RL Action and Reward
action_col = 'drug_name' # This will be the action
reward_cols = ['measure_of_response', 'vital_status', 'survival_days', 'new_tumor_event_after_initial_treatment'] # For reward calculation
time_col = 'days_to_drug_therapy_start' # For ordering trajectories
patient_id_col = 'bcr_patient_barcode' # Patient identifier

# Ensure essential columns are present
for col in [patient_id_col, time_col, action_col] + reward_cols:
    if col not in final_df_for_rl.columns:
        print(f"Error: Essential column '{col}' missing from final DataFrame. Cannot proceed.")
        exit()

# 3. Encode Categorical Features and Scale Numerical Features for the State
# Identify numerical and categorical features from the final state_features list
numerical_features_for_scaling = final_df_for_rl[state_features].select_dtypes(include=np.number).columns.tolist()
categorical_features_for_encoding = final_df_for_rl[state_features].select_dtypes(include='object').columns.tolist()

# Create preprocessing pipelines for numerical and categorical features
# Define actual transformers
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

# Create a ColumnTransformer to apply different transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features_for_scaling),
        ('cat', categorical_transformer, categorical_features_for_encoding)
    ],
    remainder='passthrough' # Keep other columns (like gene features which are already 0/1)
)

# Apply transformations to the selected state features
# Temporarily combine state features with patient_id and time_col for processing
df_to_transform = final_df_for_rl[[patient_id_col, time_col] + state_features].copy()
df_to_transform_indexed = df_to_transform.set_index([patient_id_col, time_col])

# Fit the preprocessor
preprocessor.fit(df_to_transform_indexed)

# Transform the data
processed_state_features_array = preprocessor.transform(df_to_transform_indexed)

# Get feature names for the transformed data
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features_for_encoding)


all_state_feature_names = numerical_features_for_scaling + list(ohe_feature_names)

# Add back any 'passthrough' columns (which would be the selected gene features)
passthrough_cols = [col for col in state_features if col not in numerical_features_for_scaling and col not in categorical_features_for_encoding]
all_state_feature_names.extend(passthrough_cols)

# Create a DataFrame with transformed state features
processed_state_df = pd.DataFrame(processed_state_features_array, columns=all_state_feature_names, index=df_to_transform_indexed.index)
processed_state_df = processed_state_df.reset_index() # Bring back patient_id and time_col

print(f"\nProcessed State Features DataFrame. Shape: {processed_state_df.shape}")
print(processed_state_df.head())

# 4. Define Action Space and Map Actions to Integers
all_drug_combinations = final_df_for_rl[action_col].unique()
action_to_int = {action: i for i, action in enumerate(all_drug_combinations)}
int_to_action = {i: action for action, i in action_to_int.items()} # For reverse mapping

final_df_for_rl['action_id'] = final_df_for_rl[action_col].map(action_to_int)
print(f"\nDefined {len(all_drug_combinations)} unique actions (drug combinations).")

# 5. Define Reward Function
def calculate_reward(row):
    reward = 0

    # Tumor Response (from 'measure_of_response')
    # Higher positive reward for better response
    if row['measure_of_response'] == 'Complete Response':
        reward += 5
    elif row['measure_of_response'] == 'Partial Response':
        reward += 3
    elif row['measure_of_response'] == 'Stable Disease':
        reward += 1
    elif row['measure_of_response'] == 'Progressive Disease':
        reward -= 5 # Significant penalty for progression
    else: # Handle 'Unknown' or other unmapped responses
        reward -= 1 # Small penalty for unclear response

    # Survival (from 'vital_status' and 'survival_days')
    # Long-term positive reward for being alive, penalty for death.
    # Normalize survival days to contribute proportionally to reward.
    max_survival = final_df_for_rl['survival_days'].max()
    min_survival = final_df_for_rl['survival_days'].min()
    if max_survival > min_survival:
        normalized_survival = (row['survival_days'] - min_survival) / (max_survival - min_survival)
        reward += normalized_survival * 2 # Smaller weight than immediate response
    else:
        reward += 1 # Neutral if no variation

    if row['vital_status'] == 'Dead':
        reward -= 10 # Large penalty for death

    # Disease Progression/Recurrence (from 'new_tumor_event_after_initial_treatment')
    # Penalty for new tumor events
    if row['new_tumor_event_after_initial_treatment'] == 1: # Assuming 1 for YES
        reward -= 7 # Penalty for recurrence

    # Toxicity (Implicit, as direct toxicity data is limited)
    # This is a placeholder. In a real scenario, explicit toxicity grades would be used.
    # For now, we can infer some toxicity from very short treatment durations or early death/progression
    # if not already captured by other penalties.
    # This part is more conceptual for this dataset due to lack of explicit toxicity data.
    # If treatment duration is very short and outcome is poor, it might imply toxicity/intolerance.
    if row['treatment_duration_drug'] < 30 and (row['measure_of_response'] == 'Progressive Disease' or row['vital_status'] == 'Dead'):
        reward -= 2 # Small additional penalty for very short, ineffective/harmful treatment

    return reward

final_df_for_rl['reward'] = final_df_for_rl.apply(calculate_reward, axis=1)
print("Calculated 'reward' for each treatment step.")

# 6. Construct Patient Trajectories for RL Training
# Merge processed state features back with action_id, reward, and original patient_id/time_col
# This ensures all relevant data for a trajectory step is together.
rl_data_full = pd.merge(
    processed_state_df,
    final_df_for_rl[[patient_id_col, time_col, 'action_id', 'reward']],
    on=[patient_id_col, time_col],
    how='inner'
)

# Sort again to ensure correct temporal order for trajectories
rl_data_full.sort_values(by=[patient_id_col, time_col], inplace=True)

patient_trajectories = defaultdict(list)
# Iterate through each patient to build their sequence of (state, action, reward, next_state, done)
for patient_id in rl_data_full[patient_id_col].unique():
    patient_records = rl_data_full[rl_data_full[patient_id_col] == patient_id].copy()
    
    # Extract state features (all columns except patient_id, time_col, action_id, reward)
    current_state_feature_cols = [col for col in processed_state_df.columns if col not in [patient_id_col, time_col]]

    for i in range(len(patient_records)):
        current_step = patient_records.iloc[i]
        current_state = current_step[current_state_feature_cols].values
        action = current_step['action_id']
        reward = current_step['reward']
        
        next_state = None
        done = False
        if i + 1 < len(patient_records):
            next_step = patient_records.iloc[i+1]
            next_state = next_step[current_state_feature_cols].values
        else:
            done = True # This is the last observed step for this patient

        patient_trajectories[patient_id].append({
            'state': current_state,
            'action': action,
            'reward': reward,
            'next_state': next_state,
            'done': done
        })

print(f"\nConstructed {len(patient_trajectories)} patient trajectories.")
print("\nExample of the first step for the first patient's trajectory:")
if patient_trajectories:
    first_patient_id = list(patient_trajectories.keys())
    first_step = patient_trajectories[first_patient_id]
    print(f"  Patient ID: {first_patient_id}")
    print(f"  State (first 5 values): {first_step['state'][:5]}")
    print(f"  Action ID: {first_step['action']} (Drug: {int_to_action.get(first_step['action'], 'Unknown')})")
    print(f"  Reward: {first_step['reward']:.2f}")
    print(f"  Is Done: {first_step['done']}")
    if first_step['next_state'] is not None:
        print(f"  Next State (first 5 values): {first_step['next_state'][:5]}")
else:
    print("No trajectories could be constructed. Check data integrity and merging steps.")

print("\n--- Preprocessing Complete ---")


--- Phase 4: Feature Selection and Final Data Preparation for RL ---
Reduced gene features from 101 to 2 using Variance Thresholding.


NotFittedError: This OneHotEncoder instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [9]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Assume 'merged_df' and 'gene_df_transposed' are pre-loaded DataFrames.
# For reproducibility, let's create dummy DataFrames.
# You should replace this with your actual data loading.
dummy_clinical_data = {
    'bcr_patient_barcode': ['P1', 'P1', 'P2', 'P3', 'P3', 'P3'],
    'drug_name': ['Drug A', 'Drug B', 'Drug A', 'Drug C', 'Drug A', 'Drug B'],
    'measure_of_response': ['Complete Response', 'Progressive Disease', 'Partial Response', 'Stable Disease', 'Unknown', 'Complete Response'],
    'vital_status': ['Alive', 'Dead', 'Alive', 'Alive', 'Dead', 'Alive'],
    'survival_days': [800, 250, 600, 900, 150, 1200],
    'new_tumor_event_after_initial_treatment': [0, 1, 0, 0, 1, 0],
    'days_to_drug_therapy_start': [0, 260, 0, 0, 150, 400],
    'age_at_diagnosis': [55, 55, 62, 48, 70, 70],
    'ajcc_pathologic_tumor_stage': ['Stage II', 'Stage II', 'Stage III', 'Stage I', 'Stage IV', 'Stage IV']
}
dummy_gene_data = {
    'bcr_patient_barcode': ['P1', 'P2', 'P3'],
    'GENE_A': [1, 0, 1],
    'GENE_B': [0, 0, 1],
    'GENE_C': [1, 1, 1], # Low variance (present in all)
    'GENE_D': [0, 0, 0]  # Low variance (present in none)
}
merged_df = pd.merge(pd.DataFrame(dummy_clinical_data), pd.DataFrame(dummy_gene_data), on='bcr_patient_barcode', how='left')
gene_df_transposed = pd.DataFrame(dummy_gene_data).set_index('bcr_patient_barcode').T.reset_index().rename(columns={'index': 'gene'})
# --- End of Dummy Data ---


print("\n--- Phase 4: Feature Selection and Final Data Preparation for RL ---")

# 1. Feature Selection for Gene Mutations
# Identify gene mutation columns from the original gene dataframe structure.
gene_mutation_cols = [col for col in gene_df_transposed.columns if col not in ['bcr_patient_barcode', 'gene']]
gene_mutation_cols_in_merged = [col for col in gene_mutation_cols if col in merged_df.columns]

# Apply Variance Thresholding to gene mutation features
# A threshold of 0.05 removes features present in <5% or >95% of samples.
selector = VarianceThreshold(threshold=0.05)
merged_df[gene_mutation_cols_in_merged] = merged_df[gene_mutation_cols_in_merged].apply(pd.to_numeric, errors='coerce').fillna(0)

# Fit and transform only the gene mutation columns
gene_features_selected_array = selector.fit_transform(merged_df[gene_mutation_cols_in_merged])

# Get names of selected gene features
selected_gene_cols = merged_df[gene_mutation_cols_in_merged].columns[selector.get_support()]
print(f"Reduced gene features from {len(gene_mutation_cols_in_merged)} to {len(selected_gene_cols)} using Variance Thresholding.")

# Create a DataFrame for selected gene features
df_selected_genes = pd.DataFrame(gene_features_selected_array, columns=selected_gene_cols, index=merged_df.index)

# Drop original gene columns and concatenate with selected ones
final_df_for_rl = merged_df.drop(columns=gene_mutation_cols_in_merged, errors='ignore')
final_df_for_rl = pd.concat([final_df_for_rl, df_selected_genes], axis=1)

# 2. Define Features for RL State, Action, and Reward
# ** FIX: Corrected state feature definition and added placeholder clinical features. **
# ** NOTE: Replace placeholder column names with your actual clinical feature columns. **
clinical_features_for_state = ['age_at_diagnosis', 'ajcc_pathologic_tumor_stage']
state_features_candidate = clinical_features_for_state + list(selected_gene_cols)

# Filter to only include columns that actually exist in the DataFrame
state_features = [col for col in state_features_candidate if col in final_df_for_rl.columns]

# Columns for RL Action and Reward
action_col = 'drug_name'
reward_cols = ['measure_of_response', 'vital_status', 'survival_days', 'new_tumor_event_after_initial_treatment']
time_col = 'days_to_drug_therapy_start'
patient_id_col = 'bcr_patient_barcode'

# Ensure essential columns are present
for col in [patient_id_col, time_col, action_col] + reward_cols:
    if col not in final_df_for_rl.columns:
        print(f"Error: Essential column '{col}' missing from final DataFrame. Cannot proceed.")
        exit()

# 3. Encode Categorical Features and Scale Numerical Features for the State
# Identify numerical and categorical features from the final state_features list
# Note: Gene features are already binary (0/1) and will be passed through without scaling.
numerical_features_for_scaling = final_df_for_rl[state_features].select_dtypes(include=np.number).columns.tolist()
# Exclude binary gene features from scaling
numerical_features_for_scaling = [col for col in numerical_features_for_scaling if col not in selected_gene_cols]
categorical_features_for_encoding = final_df_for_rl[state_features].select_dtypes(include=['object', 'category']).columns.tolist()

# ** FIX: Completed the pipeline definitions with transformers. **
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Create a ColumnTransformer to apply different transformations
# Numerical features are scaled, categorical are one-hot encoded, and the rest (gene features) are passed through.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features_for_scaling),
        ('cat', categorical_transformer, categorical_features_for_encoding)
    ],
    remainder='passthrough' # This will keep the gene features as they are
)

# Apply transformations to the selected state features
df_to_transform = final_df_for_rl[[patient_id_col, time_col] + state_features].copy()
df_to_transform_indexed = df_to_transform.set_index([patient_id_col, time_col])

# Fit and transform the data
processed_state_features_array = preprocessor.fit_transform(df_to_transform_indexed)

# Get feature names for the transformed data
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features_for_encoding)
all_state_feature_names = numerical_features_for_scaling + list(ohe_feature_names)

# Add back any 'passthrough' columns (which are the selected gene features)
passthrough_cols = [col for col in df_to_transform_indexed.columns if col not in numerical_features_for_scaling and col not in categorical_features_for_encoding]
all_state_feature_names.extend(passthrough_cols)

# Create a DataFrame with transformed state features
processed_state_df = pd.DataFrame(processed_state_features_array, columns=all_state_feature_names, index=df_to_transform_indexed.index)
processed_state_df = processed_state_df.reset_index()

print(f"\nProcessed State Features DataFrame. Shape: {processed_state_df.shape}")
print(processed_state_df.head())

# 4. Define Action Space and Map Actions to Integers
all_drug_combinations = final_df_for_rl[action_col].unique()
action_to_int = {action: i for i, action in enumerate(all_drug_combinations)}
int_to_action = {i: action for action, i in action_to_int.items()}

final_df_for_rl['action_id'] = final_df_for_rl[action_col].map(action_to_int)
print(f"\nDefined {len(all_drug_combinations)} unique actions (drug combinations).")

# 5. Define Reward Function
def calculate_reward(row, max_survival, min_survival):
    reward = 0
    # Tumor Response
    if row['measure_of_response'] == 'Complete Response':   reward += 5
    elif row['measure_of_response'] == 'Partial Response':  reward += 3
    elif row['measure_of_response'] == 'Stable Disease':    reward += 1
    elif row['measure_of_response'] == 'Progressive Disease': reward -= 5
    else: reward -= 1 # Penalty for 'Unknown'

    # Survival
    if max_survival > min_survival:
        normalized_survival = (row['survival_days'] - min_survival) / (max_survival - min_survival)
        reward += normalized_survival * 2
    else:
        reward += 1

    if row['vital_status'] == 'Dead':
        reward -= 10

    # Disease Recurrence
    if row['new_tumor_event_after_initial_treatment'] == 1:
        reward -= 7

    # ** FIX: Commented out logic referencing a non-existent column to prevent KeyError. **
    # ** NOTE: If you have treatment duration data, you can uncomment this section. **
    # if 'treatment_duration_drug' in row and row['treatment_duration_drug'] < 30 and (row['measure_of_response'] == 'Progressive Disease' or row['vital_status'] == 'Dead'):
    #     reward -= 2

    return reward

# Calculate min/max survival once for efficiency
max_s = final_df_for_rl['survival_days'].max()
min_s = final_df_for_rl['survival_days'].min()
final_df_for_rl['reward'] = final_df_for_rl.apply(lambda row: calculate_reward(row, max_s, min_s), axis=1)
print("Calculated 'reward' for each treatment step.")

# 6. Construct Patient Trajectories for RL Training
rl_data_full = pd.merge(
    processed_state_df,
    final_df_for_rl[[patient_id_col, time_col, 'action_id', 'reward']],
    on=[patient_id_col, time_col],
    how='inner'
)
rl_data_full.sort_values(by=[patient_id_col, time_col], inplace=True)

patient_trajectories = defaultdict(list)
for patient_id in rl_data_full[patient_id_col].unique():
    patient_records = rl_data_full[rl_data_full[patient_id_col] == patient_id].copy()
    current_state_feature_cols = [col for col in processed_state_df.columns if col not in [patient_id_col, time_col]]

    for i in range(len(patient_records)):
        current_step = patient_records.iloc[i]
        current_state = current_step[current_state_feature_cols].values
        action = current_step['action_id']
        reward = current_step['reward']
        
        next_state = None
        done = False
        if i + 1 < len(patient_records):
            next_step = patient_records.iloc[i+1]
            next_state = next_step[current_state_feature_cols].values
        else:
            done = True # Last observed step for this patient

        patient_trajectories[patient_id].append({
            'state': current_state,
            'action': action,
            'reward': reward,
            'next_state': next_state,
            'done': done
        })

print(f"\nConstructed {len(patient_trajectories)} patient trajectories.")

# ** FIX: Corrected the logic for accessing and printing an example trajectory. **
print("\nExample of the first step for the first patient's trajectory:")
if patient_trajectories:
    # Get the ID of the first patient
    first_patient_id = list(patient_trajectories.keys())[0]
    # Get the full trajectory for this patient (a list of steps)
    first_trajectory = patient_trajectories[first_patient_id]
    
    if first_trajectory:
        # Get the first step (a dictionary) from that trajectory
        first_step = first_trajectory[0]
        
        print(f"  Patient ID: {first_patient_id}")
        print(f"  State (first 5 values): {first_step['state'][:5]}")
        print(f"  Action ID: {first_step['action']} (Drug: {int_to_action.get(first_step['action'], 'Unknown')})")
        print(f"  Reward: {first_step['reward']:.2f}")
        print(f"  Is Done: {first_step['done']}")
        if first_step['next_state'] is not None:
            print(f"  Next State (first 5 values): {first_step['next_state'][:5]}")
    else:
        print(f"Patient {first_patient_id} has an empty trajectory.")
else:
    print("No trajectories could be constructed. Check data integrity and merging steps.")

print("\n--- Preprocessing Complete ---")


--- Phase 4: Feature Selection and Final Data Preparation for RL ---


ValueError: at least one array or dtype is required

In [10]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Assume 'merged_df' is a pre-loaded DataFrame.
# For reproducibility, we'll create the same dummy data.
# You should replace this with your actual data loading.
dummy_clinical_data = {
    'bcr_patient_barcode': ['P1', 'P1', 'P2', 'P3', 'P3', 'P3'],
    'drug_name': ['Drug A', 'Drug B', 'Drug A', 'Drug C', 'Drug A', 'Drug B'],
    'measure_of_response': ['Complete Response', 'Progressive Disease', 'Partial Response', 'Stable Disease', 'Unknown', 'Complete Response'],
    'vital_status': ['Alive', 'Dead', 'Alive', 'Alive', 'Dead', 'Alive'],
    'survival_days': [800, 250, 600, 900, 150, 1200],
    'new_tumor_event_after_initial_treatment': [0, 1, 0, 0, 1, 0],
    'days_to_drug_therapy_start': [0, 260, 0, 0, 150, 400],
    'age_at_diagnosis': [55, 55, 62, 48, 70, 70],
    'ajcc_pathologic_tumor_stage': ['Stage II', 'Stage II', 'Stage III', 'Stage I', 'Stage IV', 'Stage IV']
}
dummy_gene_df = pd.DataFrame({
    'bcr_patient_barcode': ['P1', 'P2', 'P3'],
    'GENE_A': [1, 0, 1],
    'GENE_B': [0, 0, 1],
    'GENE_C': [1, 1, 1], # Low variance (present in all)
    'GENE_D': [0, 0, 0]  # Low variance (present in none)
})
merged_df = pd.merge(pd.DataFrame(dummy_clinical_data), dummy_gene_df, on='bcr_patient_barcode', how='left')
# --- End of Dummy Data ---


print("\n--- Phase 4: Feature Selection and Final Data Preparation for RL ---")

# 1. Feature Selection for Gene Mutations
# ** FIX: Correctly identify gene columns directly from the source DataFrame. **
# The previous method incorrectly derived patient IDs instead of gene names.
original_gene_cols = [col for col in dummy_gene_df.columns if col != 'bcr_patient_barcode']
gene_mutation_cols_in_merged = [col for col in original_gene_cols if col in merged_df.columns]

# This check prevents the error if no gene columns are found.
if not gene_mutation_cols_in_merged:
    print("No gene mutation columns found in the merged DataFrame. Skipping feature selection.")
    selected_gene_cols = []
    final_df_for_rl = merged_df.copy()
else:
    # Apply Variance Thresholding to gene mutation features
    selector = VarianceThreshold(threshold=0.05)
    merged_df[gene_mutation_cols_in_merged] = merged_df[gene_mutation_cols_in_merged].apply(pd.to_numeric, errors='coerce').fillna(0)

    # Fit and transform only the gene mutation columns
    gene_features_selected_array = selector.fit_transform(merged_df[gene_mutation_cols_in_merged])

    # Get names of selected gene features
    selected_gene_cols = merged_df[gene_mutation_cols_in_merged].columns[selector.get_support()]
    print(f"Reduced gene features from {len(gene_mutation_cols_in_merged)} to {len(selected_gene_cols)} using Variance Thresholding.")

    # Create a DataFrame for selected gene features
    df_selected_genes = pd.DataFrame(gene_features_selected_array, columns=selected_gene_cols, index=merged_df.index)

    # Drop original gene columns and concatenate with selected ones
    final_df_for_rl = merged_df.drop(columns=gene_mutation_cols_in_merged, errors='ignore')
    final_df_for_rl = pd.concat([final_df_for_rl, df_selected_genes], axis=1)

# 2. Define Features for RL State, Action, and Reward
# NOTE: Replace placeholder column names with your actual clinical feature columns.
clinical_features_for_state = ['age_at_diagnosis', 'ajcc_pathologic_tumor_stage']
state_features_candidate = clinical_features_for_state + list(selected_gene_cols)

# Filter to only include columns that actually exist in the DataFrame
state_features = [col for col in state_features_candidate if col in final_df_for_rl.columns]

# Columns for RL Action and Reward
action_col = 'drug_name'
reward_cols = ['measure_of_response', 'vital_status', 'survival_days', 'new_tumor_event_after_initial_treatment']
time_col = 'days_to_drug_therapy_start'
patient_id_col = 'bcr_patient_barcode'

# Ensure essential columns are present
for col in [patient_id_col, time_col, action_col] + reward_cols:
    if col not in final_df_for_rl.columns:
        print(f"Error: Essential column '{col}' missing from final DataFrame. Cannot proceed.")
        exit()

# 3. Encode Categorical Features and Scale Numerical Features for the State
numerical_features_for_scaling = final_df_for_rl[state_features].select_dtypes(include=np.number).columns.tolist()
numerical_features_for_scaling = [col for col in numerical_features_for_scaling if col not in selected_gene_cols]
categorical_features_for_encoding = final_df_for_rl[state_features].select_dtypes(include=['object', 'category']).columns.tolist()

numerical_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features_for_scaling),
        ('cat', categorical_transformer, categorical_features_for_encoding)
    ],
    remainder='passthrough'
)

# Apply transformations
df_to_transform = final_df_for_rl[[patient_id_col, time_col] + state_features].copy()
df_to_transform_indexed = df_to_transform.set_index([patient_id_col, time_col])

processed_state_features_array = preprocessor.fit_transform(df_to_transform_indexed)

# Get feature names for the transformed data
try:
    ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features_for_encoding)
except AttributeError: # For older scikit-learn versions
    ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names(categorical_features_for_encoding)

all_state_feature_names = numerical_features_for_scaling + list(ohe_feature_names)
passthrough_cols = [col for col in df_to_transform_indexed.columns if col not in numerical_features_for_scaling and col not in categorical_features_for_encoding]
all_state_feature_names.extend(passthrough_cols)

# Create a DataFrame with transformed state features
processed_state_df = pd.DataFrame(processed_state_features_array, columns=all_state_feature_names, index=df_to_transform_indexed.index)
processed_state_df = processed_state_df.reset_index()

print(f"\nProcessed State Features DataFrame. Shape: {processed_state_df.shape}")
print(processed_state_df.head())

# 4. Define Action Space and Map Actions to Integers
all_drug_combinations = final_df_for_rl[action_col].dropna().unique()
action_to_int = {action: i for i, action in enumerate(all_drug_combinations)}
int_to_action = {i: action for action, i in action_to_int.items()}

final_df_for_rl['action_id'] = final_df_for_rl[action_col].map(action_to_int)
print(f"\nDefined {len(all_drug_combinations)} unique actions (drug combinations).")

# 5. Define Reward Function
def calculate_reward(row, max_survival, min_survival):
    reward = 0
    if row['measure_of_response'] == 'Complete Response':   reward += 5
    elif row['measure_of_response'] == 'Partial Response':  reward += 3
    elif row['measure_of_response'] == 'Stable Disease':    reward += 1
    elif row['measure_of_response'] == 'Progressive Disease': reward -= 5
    else: reward -= 1 # Penalty for 'Unknown'

    if max_survival > min_survival and pd.notna(row['survival_days']):
        normalized_survival = (row['survival_days'] - min_survival) / (max_survival - min_survival)
        reward += normalized_survival * 2
    else:
        reward += 1

    if row['vital_status'] == 'Dead':
        reward -= 10

    if pd.notna(row['new_tumor_event_after_initial_treatment']) and row['new_tumor_event_after_initial_treatment'] == 1:
        reward -= 7

    return reward

max_s = final_df_for_rl['survival_days'].max()
min_s = final_df_for_rl['survival_days'].min()
final_df_for_rl['reward'] = final_df_for_rl.apply(lambda row: calculate_reward(row, max_s, min_s), axis=1)
print("Calculated 'reward' for each treatment step.")

# 6. Construct Patient Trajectories for RL Training
rl_data_full = pd.merge(
    processed_state_df,
    final_df_for_rl[[patient_id_col, time_col, 'action_id', 'reward']],
    on=[patient_id_col, time_col],
    how='inner'
)
rl_data_full.sort_values(by=[patient_id_col, time_col], inplace=True)

patient_trajectories = defaultdict(list)
for patient_id in rl_data_full[patient_id_col].unique():
    patient_records = rl_data_full[rl_data_full[patient_id_col] == patient_id].copy()
    current_state_feature_cols = [col for col in processed_state_df.columns if col not in [patient_id_col, time_col]]

    for i in range(len(patient_records)):
        current_step = patient_records.iloc[i]
        current_state = current_step[current_state_feature_cols].values
        action = current_step['action_id']
        reward = current_step['reward']
        
        next_state = None
        done = False
        if i + 1 < len(patient_records):
            next_step = patient_records.iloc[i+1]
            next_state = next_step[current_state_feature_cols].values
        else:
            done = True

        patient_trajectories[patient_id].append({
            'state': current_state, 'action': action, 'reward': reward,
            'next_state': next_state, 'done': done
        })

print(f"\nConstructed {len(patient_trajectories)} patient trajectories.")
print("\nExample of the first step for the first patient's trajectory:")
if patient_trajectories:
    first_patient_id = list(patient_trajectories.keys())[0]
    first_trajectory = patient_trajectories[first_patient_id]
    
    if first_trajectory:
        first_step = first_trajectory[0]
        print(f"  Patient ID: {first_patient_id}")
        print(f"  State (first 5 values): {first_step['state'][:5]}")
        print(f"  Action ID: {first_step['action']} (Drug: {int_to_action.get(first_step['action'], 'Unknown')})")
        print(f"  Reward: {first_step['reward']:.2f}")
        print(f"  Is Done: {first_step['done']}")
        if first_step['next_state'] is not None:
            print(f"  Next State (first 5 values): {first_step['next_state'][:5]}")
    else:
        print(f"Patient {first_patient_id} has an empty trajectory.")
else:
    print("No trajectories could be constructed. Check data integrity.")

print("\n--- Preprocessing Complete ---")


--- Phase 4: Feature Selection and Final Data Preparation for RL ---
Reduced gene features from 4 to 2 using Variance Thresholding.

Processed State Features DataFrame. Shape: (6, 9)
  bcr_patient_barcode  days_to_drug_therapy_start  age_at_diagnosis  \
0                  P1                           0         -0.613909   
1                  P1                         260         -0.613909   
2                  P2                           0          0.245564   
3                  P3                           0         -1.473382   
4                  P3                         150          1.227818   

   ajcc_pathologic_tumor_stage_Stage I  ajcc_pathologic_tumor_stage_Stage II  \
0                                  0.0                                   1.0   
1                                  0.0                                   1.0   
2                                  0.0                                   0.0   
3                                  1.0                               

In [11]:
final_df_for_rl.to_excel('final_rl_dataframe.excel', index=False)
print("Saved the main data processing steps to 'final_rl_dataframe.csv'")

ValueError: No engine for filetype: 'excel'

In [12]:
final_df_for_rl.to_csv('final_rl_dataframe.csv', index=False)
print("Saved the main data processing steps to 'final_rl_dataframe.csv'")

Saved the main data processing steps to 'final_rl_dataframe.csv'


In [1]:
print(final_rl_df['bcr_patient_barcode'].nunique())  # How many unique patients
print(final_rl_df.shape[0])  # How many treatment records (episodes)

NameError: name 'final_rl_df' is not defined

In [2]:
print(final_rl_dataframe['bcr_patient_barcode'].nunique())  # How many unique patients
print(final_rl_dataframe.shape[0])  # How many treatment records (episodes)

NameError: name 'final_rl_dataframe' is not defined